# IPL Live Win Predictor 🏆

**Goal:** train a model that, at any ball during the chase (innings 2), predicts the chasing team's win probability — exactly like the win-probability bar you see on TV.

**Approach:**
1. Build features from the ball-by-ball data (score so far, wickets left, balls left, required run rate, venue, teams)
2. Train models on innings-2 deliveries
3. Evaluate, compare, and visualize
4. Make a live prediction function
5. Use it to forecast this year's season

**Prerequisite:** run `get_ipl_data.py` first so the CSVs exist.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, log_loss,
                              classification_report, confusion_matrix,
                              roc_curve, brier_score_loss)

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
DATA_DIR = 'ipl_data'

deliveries = pd.read_csv(f'{DATA_DIR}/deliveries_features.csv')
matches    = pd.read_csv(f'{DATA_DIR}/matches.csv')

deliveries['match_id'] = deliveries['match_id'].astype(str)
matches['match_id']    = matches['match_id'].astype(str)

print('Deliveries:', deliveries.shape)
print('Matches:   ', matches.shape)

## 2. Build the Training Set

We use **innings-2 deliveries** so we have a target (the chasing team's batting_team is the candidate winner). The label is *did the batting (chasing) team win this match?*

In [ ]:
# First-innings totals -> the target the chaser must beat
first_inn = (deliveries[deliveries['innings'] == 1]
             .groupby('match_id')['total_runs'].sum()
             .reset_index().rename(columns={'total_runs': 'first_innings_total'}))

df = deliveries[deliveries['innings'] == 2].copy()
df = df.merge(first_inn, on='match_id', how='inner')
df = df.merge(matches[['match_id', 'winner']], on='match_id', how='left')

# Drop matches without a clean winner (no-results, super-overs unresolved)
df = df.dropna(subset=['winner'])

# Engineer chase features
df['target']        = df['first_innings_total'] + 1
df['runs_left']     = (df['target'] - df['current_score']).clip(lower=0)
df['balls_left']    = df['balls_remaining']
df['wickets_in_hand'] = (10 - df['current_wickets']).clip(lower=0)
df['crr']           = df['current_run_rate'].fillna(0)
df['rrr']           = (df['runs_left'] / df['balls_left'].replace(0, np.nan) * 6).fillna(99)
df['rrr']           = df['rrr'].clip(upper=36)  # cap unrealistic values when 1 ball left
df['run_rate_diff'] = df['crr'] - df['rrr']
df['pressure']      = df['runs_left'] / df['wickets_in_hand'].replace(0, np.nan)
df['pressure']      = df['pressure'].replace([np.inf, -np.inf], df['runs_left']).fillna(df['runs_left'])

# Label
df['chase_won'] = (df['batting_team'] == df['winner']).astype(int)

# Keep only meaningful rows (some Cricsheet rows have ball numbering oddities)
df = df[df['balls_left'] >= 0].copy()
df = df[df['balls_left'] <= 120].copy()

print('Training rows:', len(df))
print('Chase-won rate (label balance):', df['chase_won'].mean().round(3))
df[['match_id', 'over_num', 'current_score', 'target', 'runs_left',
    'balls_left', 'wickets_in_hand', 'crr', 'rrr', 'chase_won']].head()

## 3. Train/Test Split — by match, not by ball

**Critical:** if we split rows randomly, balls from the same match leak across train/test and we'll get a falsely high score. We split by `match_id` so the model never sees a match in both sets.

In [ ]:
match_ids = df['match_id'].unique()
train_ids, test_ids = train_test_split(match_ids, test_size=0.2, random_state=RANDOM_STATE)

train_df = df[df['match_id'].isin(train_ids)].copy()
test_df  = df[df['match_id'].isin(test_ids)].copy()

print(f'Matches: {len(match_ids)}  -> train {len(train_ids)} / test {len(test_ids)}')
print(f'Rows:    train {len(train_df):,} / test {len(test_df):,}')

In [ ]:
NUM_FEATURES = [
    'runs_left', 'balls_left', 'wickets_in_hand',
    'crr', 'rrr', 'run_rate_diff', 'pressure',
    'current_score', 'current_wickets', 'target',
    'runs_last_30_balls', 'wickets_last_30_balls'
]
CAT_FEATURES = ['batting_team', 'bowling_team', 'venue']

FEATURES = NUM_FEATURES + CAT_FEATURES
TARGET   = 'chase_won'

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]

# Some venue/team values might appear in test but not train — handle gracefully
preprocessor = ColumnTransformer([
    ('num', 'passthrough', NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES),
])
print('Numeric:', NUM_FEATURES)
print('Categorical:', CAT_FEATURES)

## 4. Train Three Models and Compare

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=12,
                                                  min_samples_leaf=20, n_jobs=-1,
                                                  random_state=RANDOM_STATE),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, max_depth=5,
                                                     learning_rate=0.05,
                                                     random_state=RANDOM_STATE)
}

results = []
pipelines = {}

for name, clf in models.items():
    print(f'Training {name} ...')
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe

    proba = pipe.predict_proba(X_test)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    results.append({
        'Model':    name,
        'Accuracy': accuracy_score(y_test, pred),
        'AUC':      roc_auc_score(y_test, proba),
        'LogLoss':  log_loss(y_test, proba),
        'Brier':    brier_score_loss(y_test, proba),
    })

results_df = pd.DataFrame(results).round(4)
print()
print(results_df.to_string(index=False))

In [ ]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=('Accuracy ↑', 'AUC ↑', 'Log Loss ↓ (lower=better)',
                    'Brier Score ↓ (lower=better)'))
metrics = [('Accuracy', 1, 1), ('AUC', 1, 2), ('LogLoss', 2, 1), ('Brier', 2, 2)]
colors  = ['#3498db', '#e74c3c', '#2ecc71']
for m, r, c in metrics:
    fig.add_trace(go.Bar(
        x=results_df['Model'], y=results_df[m],
        text=results_df[m].round(4),
        textposition='outside',
        marker_color=colors,
        showlegend=False
    ), row=r, col=c)
fig.update_layout(height=700, title='📊 Model Comparison')
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                          line=dict(dash='dash', color='grey'),
                          name='Random'))
for name, pipe in pipelines.items():
    proba = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                              name=f'{name} (AUC={auc:.3f})'))
fig.update_layout(title='🎯 ROC Curves',
                  xaxis_title='False Positive Rate',
                  yaxis_title='True Positive Rate',
                  height=500, width=700)
fig.show()

## 5. Pick the Best Model & Check Calibration

For a win-probability model, **calibration** matters more than raw accuracy. When the model says "70%", do the team actually win ~70% of the time?

In [ ]:
# Best by log loss (calibration-friendly)
best_name  = results_df.sort_values('LogLoss').iloc[0]['Model']
best_model = pipelines[best_name]
print(f'🏆 Best model by log-loss: {best_name}')

In [ ]:
from sklearn.calibration import calibration_curve

proba = best_model.predict_proba(X_test)[:, 1]
frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10, strategy='quantile')

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines',
                         line=dict(dash='dash', color='grey'),
                         name='Perfect calibration'))
fig.add_trace(go.Scatter(x=mean_pred, y=frac_pos, mode='lines+markers',
                         marker=dict(size=10, color='#e74c3c'),
                         line=dict(color='#e74c3c'),
                         name=best_name))
fig.update_layout(title=f'📏 Calibration Curve - {best_name}',
                  xaxis_title='Predicted probability',
                  yaxis_title='Actual win rate',
                  height=500, width=700)
fig.show()

## 6. Feature Importance

In [ ]:
# Get feature names after one-hot expansion
preproc = best_model.named_steps['prep']
ohe = preproc.named_transformers_['cat']
cat_names = list(ohe.get_feature_names_out(CAT_FEATURES))
all_names = NUM_FEATURES + cat_names

clf = best_model.named_steps['clf']
if hasattr(clf, 'feature_importances_'):
    imp = clf.feature_importances_
else:
    imp = np.abs(clf.coef_[0])

fi_df = (pd.DataFrame({'feature': all_names, 'importance': imp})
         .sort_values('importance', ascending=False)
         .head(20))

fig = px.bar(fi_df, x='importance', y='feature', orientation='h',
             title=f'🔍 Top 20 Features - {best_name}',
             color='importance', color_continuous_scale='Viridis')
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'}, showlegend=False)
fig.show()

## 7. Live Win-Probability Function

Now the fun part. Given a chase situation, get a live win probability.

In [ ]:
def predict_win(batting_team, bowling_team, venue,
                target, current_score, wickets_lost, overs_completed,
                recent_runs=None, recent_wickets=None):
    '''
    Returns the chasing team's win probability (0-1).
    
    overs_completed: float like 12.3 means 12 overs and 3 balls
    '''
    overs_int = int(overs_completed)
    balls_in  = round((overs_completed - overs_int) * 10)
    legal_balls_bowled = overs_int * 6 + balls_in
    
    runs_left   = max(target - current_score, 0)
    balls_left  = max(120 - legal_balls_bowled, 0)
    wkts_hand   = max(10 - wickets_lost, 0)
    crr         = (current_score / legal_balls_bowled * 6) if legal_balls_bowled else 0
    rrr         = (runs_left / balls_left * 6) if balls_left else 36
    rrr         = min(rrr, 36)
    
    row = pd.DataFrame([{
        'runs_left': runs_left, 'balls_left': balls_left,
        'wickets_in_hand': wkts_hand, 'crr': crr, 'rrr': rrr,
        'run_rate_diff': crr - rrr,
        'pressure': runs_left / max(wkts_hand, 1),
        'current_score': current_score, 'current_wickets': wickets_lost,
        'target': target,
        'runs_last_30_balls':    recent_runs    if recent_runs    is not None else crr * 5,
        'wickets_last_30_balls': recent_wickets if recent_wickets is not None else 1,
        'batting_team': batting_team, 'bowling_team': bowling_team,
        'venue': venue
    }])
    return float(best_model.predict_proba(row)[0, 1])


# Example: CSK chasing 180 vs MI, 60/2 after 8 overs at Wankhede
p = predict_win(
    batting_team='Chennai Super Kings',
    bowling_team='Mumbai Indians',
    venue='Wankhede Stadium',
    target=180, current_score=60, wickets_lost=2, overs_completed=8.0
)
print(f'CSK win probability: {p*100:.1f}%')

## 8. Replay a Real Match Ball-by-Ball

Pick a random recent match from the test set and plot the live win-probability curve.

In [ ]:
# Pick a random match from the test set
sample_mid = np.random.RandomState(7).choice(list(test_ids))
sample = test_df[test_df['match_id'] == sample_mid].sort_values('legal_balls_bowled').copy()

sample['win_prob'] = best_model.predict_proba(sample[FEATURES])[:, 1] * 100

minfo = matches[matches['match_id'] == sample_mid].iloc[0]
actual_winner = sample.iloc[0]['winner']
chaser        = sample.iloc[0]['batting_team']
outcome       = 'WON' if chaser == actual_winner else 'LOST'

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sample['legal_balls_bowled'], y=sample['win_prob'],
    mode='lines+markers',
    line=dict(width=3, color='#3498db'),
    marker=dict(size=4),
    hovertemplate='Ball %{x}<br>Win prob: %{y:.1f}%<br>Score: %{customdata[0]}/%{customdata[1]}<extra></extra>',
    customdata=sample[['current_score', 'current_wickets']].values,
    name=chaser
))
fig.add_hline(y=50, line_dash='dash', line_color='grey')

# Mark wickets
wickets = sample[sample['is_wicket'] == 1]
if len(wickets):
    fig.add_trace(go.Scatter(
        x=wickets['legal_balls_bowled'], y=wickets['win_prob'],
        mode='markers',
        marker=dict(symbol='x', size=14, color='red'),
        name='Wicket!'
    ))

fig.update_layout(
    title=(f'🎬 Match Replay - {chaser} chasing {int(sample.iloc[0]["target"])}'
           f' at {minfo["venue"]}<br>'
           f'<sub>Final: {chaser} {outcome}</sub>'),
    xaxis_title='Balls Bowled',
    yaxis_title='Win Probability (%)',
    yaxis_range=[0, 100],
    height=500
)
fig.show()

## 9. Forecast the Current Season

Use the trained model to estimate each team's average chase win probability across all their chases this season — a rough "who's winning the title?" signal.

In [ ]:
# Latest season in the data
latest_season = sorted(matches['season'].dropna().unique())[-1]
print(f'Latest season in data: {latest_season}')

season_matches = matches[matches['season'] == latest_season]['match_id'].tolist()
season_df = df[df['match_id'].isin(season_matches)].copy()

if len(season_df) == 0:
    print('No innings-2 data for the latest season yet.')
else:
    season_df['win_prob'] = best_model.predict_proba(season_df[FEATURES])[:, 1]

    # Average end-of-innings win probability per team this season
    end_state = (season_df.sort_values('legal_balls_bowled')
                 .groupby('match_id').tail(1))
    
    # Power-rank teams: combined batting (chase WP) + bowling (1 - opponent chase WP)
    bat_rank = end_state.groupby('batting_team').agg(
        chases=('match_id', 'count'),
        avg_chase_wp=('win_prob', 'mean'),
        wins=('chase_won', 'sum')
    ).reset_index().rename(columns={'batting_team': 'team'})
    bat_rank['actual_chase_win_rate'] = bat_rank['wins'] / bat_rank['chases']
    bat_rank = bat_rank.sort_values('avg_chase_wp', ascending=False)
    print()
    print(f'🏆 Team chase strength this season ({latest_season}):')
    print(bat_rank.round(3))

    fig = px.bar(
        bat_rank,
        x='avg_chase_wp', y='team', orientation='h',
        text=bat_rank['avg_chase_wp'].round(2),
        color='avg_chase_wp', color_continuous_scale='RdYlGn',
        hover_data={'chases': True, 'wins': True, 'actual_chase_win_rate': ':.2f'},
        title=f'🏆 Model-Estimated Chase Strength - Season {latest_season}',
        labels={'avg_chase_wp': 'Avg End-of-Chase Win Probability', 'team': ''}
    )
    fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'}, showlegend=False)
    fig.show()

## 10. Save the Model

In [ ]:
import joblib
import os

os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/ipl_win_predictor.pkl')
print('Saved models/ipl_win_predictor.pkl')
print()
print('To use later:')
print('  import joblib')
print('  model = joblib.load("models/ipl_win_predictor.pkl")')
print('  model.predict_proba(your_dataframe)')

## Next steps to push this further

- **Calibration:** if the calibration curve is off, wrap the best model in `CalibratedClassifierCV`.
- **Player features:** add striker/bowler career stats — currently the model only sees team-level info.
- **Cross-validation by season:** train on seasons 2008-2023, validate on 2024, test on 2025 — more realistic time-series eval.
- **First-innings predictor:** the symmetric model — predict final 1st-innings total given current state, then use the target to feed the chase model.
- **Streamlit / Gradio app:** wrap `predict_win()` in a slider UI for an interactive demo.